# EDGE: Music-Driven Dance Generation

本 notebook 在 Google Colab 上运行 EDGE 模型，从音乐生成舞蹈动作。

**使用方法：**
1. 在 Colab 中打开（Runtime → Change runtime type → T4 GPU）
2. 上传你的 wav 音乐文件
3. 运行所有 cell
4. 下载生成的 .npy 动作文件

---

## 1. 环境安装

In [ ]:
# 检查 GPU
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 安装依赖
!pip install jukemirlib librosa soundfile einops tqdm accelerate pytorch3d -q
# 如果 pytorch3d 安装失败，用下面这行：
# !pip install "git+https://github.com/facebookresearch/pytorch3d.git" -q

In [ ]:
# 克隆 EDGE 仓库
import os
if not os.path.exists("EDGE"):
    !git clone https://github.com/Stanford-TML/EDGE.git
os.chdir("EDGE")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# 下载预训练 checkpoint
# EDGE 官方提供了 jukebox feature 版本的 checkpoint
if not os.path.exists("checkpoint.pt"):
    !gdown "https://drive.google.com/uc?id=1YNHs0JEMYBBPFMfqGSIU_ezLfmGfTjmP" -O checkpoint.pt
    # 如果 gdown 失败，手动下载：
    # https://drive.google.com/file/d/1YNHs0JEMYBBPFMfqGSIU_ezLfmGfTjmP/view
print(f"Checkpoint: {os.path.getsize('checkpoint.pt') / 1e6:.1f} MB")

## 2. 上传音乐文件

上传你的 **ballet.wav** 和 **hiphop.wav**

In [ ]:
from google.colab import files
import shutil

# 创建输入目录
os.makedirs("my_music", exist_ok=True)

print("请上传 ballet.wav 和 hiphop.wav：")
uploaded = files.upload()

for filename, content in uploaded.items():
    dest = os.path.join("my_music", filename)
    with open(dest, "wb") as f:
        f.write(content)
    print(f"  Saved: {dest} ({len(content)/1024:.0f} KB)")

print(f"\nFiles in my_music/: {os.listdir('my_music')}")

## 3. 提取音频特征（Jukebox）

EDGE 使用 Jukebox 模型提取音频特征，这是最耗时的步骤（约5-10分钟/首）。

In [ ]:
import glob
import numpy as np
from functools import cmp_to_key
from pathlib import Path
from tempfile import TemporaryDirectory

import jukemirlib
from tqdm import tqdm
from data.slice import slice_audio

FPS = 30
JUKE_LAYER = 66

def extract_jukebox_features(wav_file, out_dir):
    """Slice audio and extract Jukebox features for each slice."""
    os.makedirs(out_dir, exist_ok=True)
    songname = os.path.splitext(os.path.basename(wav_file))[0]
    save_dir = os.path.join(out_dir, songname)
    os.makedirs(save_dir, exist_ok=True)

    # Slice audio into 5-second chunks with 2.5s stride
    print(f"  Slicing {wav_file}...")
    n_slices = slice_audio(wav_file, 2.5, 5.0, save_dir)
    print(f"  Created {n_slices} slices")

    # Extract Jukebox features for each slice
    wav_slices = sorted(glob.glob(os.path.join(save_dir, "*.wav")))
    print(f"  Extracting Jukebox features...")
    for wav_slice in tqdm(wav_slices):
        feat_path = os.path.splitext(wav_slice)[0] + ".npy"
        if os.path.exists(feat_path):
            continue
        audio = jukemirlib.load_audio(wav_slice)
        reps = jukemirlib.extract(audio, layers=[JUKE_LAYER], downsample_target_rate=FPS)
        np.save(feat_path, reps[JUKE_LAYER])

    return save_dir, wav_slices

# Extract features for all uploaded music
feature_dirs = {}
for wav_file in sorted(glob.glob("my_music/*.wav")):
    name = os.path.splitext(os.path.basename(wav_file))[0]
    print(f"\nProcessing: {name}")
    save_dir, slices = extract_jukebox_features(wav_file, "cached_features")
    feature_dirs[name] = save_dir
    print(f"  Done: {len(slices)} slices")

print(f"\nFeature extraction complete!")

## 4. 生成舞蹈动作

In [ ]:
import pickle
import torch
from EDGE import EDGE
from data.audio_extraction.jukebox_features import extract as juke_extract

# 加载模型
print("Loading EDGE model...")
model = EDGE("jukebox", "checkpoint.pt")
model.eval()
print("Model loaded!")

# Sort helper
key_func = lambda x: int(os.path.splitext(x)[0].split("_")[-1].split("slice")[-1])

def stringintcmp_(a, b):
    aa, bb = "_".join(a.split("_")[:-1]), "_".join(b.split("_")[:-1])
    ka, kb = key_func(a), key_func(b)
    if aa < bb: return -1
    if aa > bb: return 1
    if ka < kb: return -1
    if ka > kb: return 1
    return 0

stringintkey = cmp_to_key(stringintcmp_)

In [ ]:
from scipy.spatial.transform import Rotation

def axis_angle_to_rot6d(aa):
    """Convert axis-angle (N, 3) to 6D rotation (N, 6)."""
    R = Rotation.from_rotvec(aa).as_matrix()  # (N, 3, 3)
    return np.concatenate([R[:, :, 0], R[:, :, 1]], axis=-1)  # (N, 6)

def convert_edge_output_to_151(smpl_poses, smpl_trans):
    """
    Convert EDGE output format to (K, 151) for our pipeline.

    Args:
        smpl_poses: (K, 72) axis-angle for 24 joints
        smpl_trans: (K, 3) root translation

    Returns:
        motion: (K, 151) = 24*6 rot6d + 3 root_trans + 4 foot_contact
    """
    K = smpl_poses.shape[0]
    joint_aa = smpl_poses.reshape(K, 24, 3)  # (K, 24, 3)

    # Convert each joint's axis-angle to 6D rotation
    joint_rot6d = np.zeros((K, 24, 6))
    for t in range(K):
        joint_rot6d[t] = axis_angle_to_rot6d(joint_aa[t])  # (24, 6)

    joint_rot6d_flat = joint_rot6d.reshape(K, 144)  # (K, 144)

    # Root translation
    root_trans = smpl_trans  # (K, 3)

    # Foot contact: heuristic from ankle joint angles
    # Left ankle = joint 7, Right ankle = joint 8
    l_ankle_vel = np.linalg.norm(np.diff(joint_aa[:, 7], axis=0), axis=-1)
    r_ankle_vel = np.linalg.norm(np.diff(joint_aa[:, 8], axis=0), axis=-1)
    l_ankle_vel = np.concatenate([[0], l_ankle_vel])
    r_ankle_vel = np.concatenate([[0], r_ankle_vel])

    # Contact when velocity is low
    l_contact = (l_ankle_vel < np.median(l_ankle_vel)).astype(float)
    r_contact = (r_ankle_vel < np.median(r_ankle_vel)).astype(float)
    foot_contact = np.stack([l_contact, l_contact, r_contact, r_contact], axis=-1)  # (K, 4)

    # Assemble (K, 151)
    motion = np.concatenate([joint_rot6d_flat, root_trans, foot_contact], axis=-1)
    assert motion.shape == (K, 151), f"Expected (K, 151), got {motion.shape}"
    return motion

In [ ]:
# 对每首音乐生成舞蹈
os.makedirs("output_motions", exist_ok=True)
os.makedirs("output_motions_pkl", exist_ok=True)

OUT_LENGTH = 10  # 秒，和你的音乐长度一致
sample_size = int(OUT_LENGTH / 2.5) - 1  # number of 5s slices needed

for name, feat_dir in feature_dirs.items():
    print(f"\n{'='*50}")
    print(f"Generating dance for: {name}")
    print(f"{'='*50}")

    # Load cached features
    wav_files = sorted(glob.glob(os.path.join(feat_dir, "*.wav")), key=stringintkey)
    npy_files = sorted(glob.glob(os.path.join(feat_dir, "*.npy")), key=stringintkey)

    print(f"  Found {len(wav_files)} audio slices, {len(npy_files)} feature files")

    # Use all slices (up to sample_size)
    n_use = min(len(npy_files), sample_size)
    cond_list = [np.load(f) for f in npy_files[:n_use]]
    file_list = wav_files[:n_use]

    cond = torch.from_numpy(np.array(cond_list))
    print(f"  Using {n_use} slices, cond shape: {cond.shape}")

    # Generate!
    print(f"  Running diffusion (this takes ~2 minutes)...")
    data_tuple = (None, cond, file_list)
    model.render_sample(
        data_tuple,
        label=name,
        render_dir="renders",
        render_count=-1,
        fk_out="output_motions_pkl",
        render=False,  # 不渲染视频（我们只要动作数据）
    )

    # Find the output pkl file
    pkl_files = glob.glob(os.path.join("output_motions_pkl", f"{name}*.pkl"))
    if not pkl_files:
        pkl_files = glob.glob(os.path.join("output_motions_pkl", "*.pkl"))

    for pkl_file in pkl_files:
        data = pickle.load(open(pkl_file, "rb"))
        smpl_poses = data["smpl_poses"]  # (K, 72)
        smpl_trans = data["smpl_trans"]  # (K, 3)
        print(f"  EDGE output: poses={smpl_poses.shape}, trans={smpl_trans.shape}")

        # Convert to our (K, 151) format
        motion_151 = convert_edge_output_to_151(smpl_poses, smpl_trans)
        out_path = os.path.join("output_motions", f"{name}_motion.npy")
        np.save(out_path, motion_151)
        print(f"  Saved: {out_path} shape={motion_151.shape}")

        # Also save raw pkl for reference
        raw_path = os.path.join("output_motions", f"{name}_raw.pkl")
        pickle.dump(data, open(raw_path, "wb"))
        break  # Only take the first output per song

print("\n\nDone! All motions generated.")
print("Files in output_motions/:")
for f in sorted(os.listdir("output_motions")):
    size = os.path.getsize(os.path.join("output_motions", f))
    print(f"  {f}: {size/1024:.0f} KB")

## 5. 下载生成的动作文件

下载 `ballet_motion.npy` 和 `hiphop_motion.npy`，放到本地项目的 `outputs/generated_motions/` 目录下。

In [ ]:
from google.colab import files

# 打包下载
!cd output_motions && tar czf /tmp/edge_motions.tar.gz *.npy
files.download("/tmp/edge_motions.tar.gz")
print("\n下载完成！")
print("解压后把 *_motion.npy 文件放到本地项目的 outputs/generated_motions/ 目录")
print("然后在本地运行：")
print("  cd project/stage2_retarget")
print("  conda activate sim")
print("  python retarget_smplsim.py")
print("  cd ../stage3_simulation")
print("  python test_full_physics.py")
print("  python make_demo_videos.py")

## 6. (可选) 快速预览生成质量

在 Colab 上快速可视化生成的 skeleton 动作。

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_skeleton_frame(full_pose, frame_idx, title=""):
    """Plot a single skeleton frame in 3D."""
    parents = [-1, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 9, 9, 12, 13, 14, 16, 17, 18, 19, 20, 21]
    joints = full_pose[frame_idx]  # (24, 3)

    fig = plt.figure(figsize=(6, 8))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(joints[:, 0], joints[:, 1], joints[:, 2], c='blue', s=30)
    for i, p in enumerate(parents):
        if p >= 0:
            ax.plot([joints[i, 0], joints[p, 0]],
                    [joints[i, 1], joints[p, 1]],
                    [joints[i, 2], joints[p, 2]], 'b-', linewidth=2)

    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_zlim(0, 2)
    ax.set_title(title)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    plt.tight_layout()
    plt.show()

# Preview each generated motion
for pkl_file in sorted(glob.glob("output_motions/*_raw.pkl")):
    name = os.path.basename(pkl_file).replace("_raw.pkl", "")
    data = pickle.load(open(pkl_file, "rb"))
    full_pose = data["full_pose"]  # (T, 24, 3)
    T = len(full_pose)
    print(f"\n{name}: {T} frames ({T/30:.1f}s)")

    # Show 4 sample frames
    for idx in [0, T//4, T//2, 3*T//4]:
        plot_skeleton_frame(full_pose, idx, f"{name} - frame {idx}/{T}")